# Atividade 2 – Preço de Vendas de Imóveis da Terracap em Licitação Pública

### Grupo:
* Gustavo Salvador Ferraz Ferreira
* Shenia Rocha Ladeira
* Aliendres Souto Souza 
### Disciplina: Introdução ao Machine Learning 
* Prof. Dra. Roberta Moreira Wichmann
* 1º Bimestre de 2026


## 1. Introdução


* ### 1.1 **Fonte dos dados:** 

* A base de dados utilizada neste estudo foi construída a partir da recuperação de dados públicos relativos à venda de imóveis da Terracap em licitações públicas, no período de 2020 a 2026.

* Foram extraídos todos os registros de vendas efetivamente realizadas em licitações públicas, considerando apenas os itens imobiliários classificados na modalidade de venda — tanto à vista quanto a prazo — e excluindo-se os imóveis comercializados por meio das modalidades de concessão, aluguel ou taxa. Além disso, foram incorporadas variáveis complementares provenientes de outras bases institucionais, tais como a quantidade de propostas recebidas por cada imóvel e a situação do imóvel registrada na última vistoria obrigatória, contemplando condições como ocupado, obstruído, cercado, vago, entre outras.


* ### 1.2 **Contextualização:** 

* A base é relevante para o presente estudo por permitir a análise das vendas de imóveis da Terracap sob diferentes perspectivas, incluindo a evolução temporal das vendas, a distribuição por Regiões Administrativas do Distrito Federal e a influência de características dos imóveis, como dimensão, destinação de uso e situação da vistoria. Além disso, a base possibilita a análise de indicadores derivados, como o ágio absoluto — definido como a diferença entre o valor de venda e o valor de avaliação — e o ágio percentual.

* ### 1.3 **Objetivo da utilização:** 

* Analisar as vendas de lotes/terrenos da Terracap em licitações públicas, para identificar situações onde a venda é feita sem ágio, ou identificar os maiores ágios.

* ### 1.4 **Problema de pesquisa:** 

* Entre 2020 e 2026, foram comercializados 2998 imóveis pela empresa. O resultado de vendas revelou variações no preço de vendas de até 1.046%, em relação à avaliação inicial. Nessa pesquisa, chamaremos a variação entre o valor de avaliação e o valor de venda de ágio. Embora quanto maior o ágio, melhor o resultado financeiro para a empresa, quando a empresa não consegue estimar esse ágio, ela compromete a elaboração do fluxo de caixa futuro e se vê obrigada a adotar uma postura mais conservadora em relação à aplicação dos recursos financeiros presentes e futuros.

* Diante desse cenário, é necessário melhorar a estimativa do valor de venda futura dos imóveis, a partir da criação de um modelo de aprendizagem supervionado, que utilize informações como área do imóvel, localização e a destinação, por exemplo, para estimar o valor de venda desses imóveis. 

* ### 1.5 **O que vai prever?** 

* O problema proposto enquadra-se em aprendizado supervisionado do tipo regressão, tendo como objetivo prever o valor final da venda do imóvel (VALOR_VENDA).
---

## 2. Análise Descritiva Preliminar
- Relembrar brevemente as análises descritivas preliminares realizadas na **Atividade 1**.
- Engenharia de variáveis (novos atributos relevantes).
- Justificativas para cada escolha.

---

In [3]:
# importação das bibliotecas necessárias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# leitura da base de dados de vendas

vendas_df = pd.read_csv("vendas_atividade2.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("vistoria_atividade2.csv", sep=";", encoding="latin-1")
destinacoes = pd.read_csv("destinacoes_classificadas.csv", sep=";", encoding="latin-1")


vendas_df = vendas_df.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)

vendas_df = vendas_df.merge(
    destinacoes[["Codigo", "Residencial", "Comercial", "Industrial", "Institucional"]],
    how="left",
    left_on="COD_DESTINACAO_IMOVEL",
    right_on="Codigo"
)

# # remove a coluna duplicada da chave
vendas_df = vendas_df.drop(columns=["CD_IMOVEL_URBANO"])
vendas_df = vendas_df.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
vendas_df = vendas_df.drop(columns=["CD_IMOVEL"])
vendas_df = vendas_df.drop(columns=["Codigo"])
vendas_df = vendas_df.drop(columns=["COD_DESTINACAO_IMOVEL"])

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
vendas_df["AGIO_ABSOLUTO"] = vendas_df["VALOR_VENDA"] - vendas_df["VALOR_LAUDO"]
vendas_df["AGIO_PERCENTUAL"] = ((vendas_df["VALOR_VENDA"] - vendas_df["VALOR_LAUDO"]) / vendas_df["VALOR_LAUDO"]) * 100

# Retirada dos imóveis com área base/construída igual a zero (tipologia Apartamentos)
condicao = (vendas_df["AREA_BASE"] == 0) & (vendas_df["AREA_MAX_CONSTR"] == 0)
vendas_df = vendas_df[~condicao] 

# colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = vendas_df.duplicated(subset=cols, keep=False)
# remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = vendas_df.loc[~mask_dup].copy()

# Foram encontrados 145 itens que constavam como items agrupados, de um total de 2998, restando 2853 registros 
print("Linhas originais:", len(vendas_df))
print("Linhas de itens agrupados:", mask_dup.sum())
# vendas_df = tabela_sem_dups
# print("Linhas após remoção de itens agrupados:", len(vendas_df))
vendas_df = vendas_df[vendas_df["VALOR_VENDA"] > 10000]
vendas_df.to_csv('dados_atividade2.csv', sep=";", index=False, encoding='utf-8')


Linhas originais: 2990
Linhas de itens agrupados: 157


## 3. Divisão dos Dados
- Estratégia de separação entre treino e teste (ex.: 70/30, 80/20, estratificação se necessário).
- Justificativa da escolha.
- Código da divisão (scikit-learn ou outra abordagem).

---

## 4. Pré-processamento dos Dados
- Limpeza de dados missing (faltantes) e inconsistentes.
- Codificação das variáveis categóricas (ex dummy, one hot encoding)
- Padronização / Normalização (quando necessário).

---

## 5. Construção e Escolha do Modelo
- Modelos testados (ex.: Regressão Logística, Árvore de Decisão, SVM, etc.).
- Critérios para seleção dos modelos.
- Códigos de treinamento e comparação inicial.

---

## 6. Otimização de Hiperparâmetros
- Técnicas utilizadas (Grid Search, Random Search, Cross-validation).
- Principais hiperparâmetros testados.
- Melhor conjunto encontrado para cada modelo.

---

## 7. Avaliação Final do Modelo
- Métricas utilizadas (ajustar conforme o tipo de problema):
  - Classificação: Acurácia, Precisão, Recall, F1-Score, AUC-ROC.
  - Regressão: RMSE, MAE, R².
- Comparação entre modelos.
- Gráficos de desempenho:
  - Curva ROC (se classificação)
  - Matriz de confusão
  - Gráfico de erros residuais (se regressão)

---

## 8. Discussão Crítica
- Interpretação dos resultados.
- Pontos fortes e fracos do modelo final.
- Potenciais melhorias futuras.
- Limitações dos dados.

---